In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

df = pd.read_csv("../data/processed/steam_features.csv")
print(df.shape)

In [ ]:
# Separate features and target
X = df.drop(columns=['success_tier'])
y = df['success_tier']

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print(y_train.value_counts())

In [ ]:
# Model 1 - Logistic Regression (baseline)
lr_model = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)
print("Logistic Regression Results:")
print(classification_report(y_test, y_pred_lr))

In [ ]:
# Model 2 - Random Forest
rf_model = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
print("Random Forest Results:")
print(classification_report(y_test, y_pred_rf))

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

xgb_model = XGBClassifier(n_estimators=200, random_state=42, eval_metric='mlogloss')
xgb_model.fit(X_train, y_train_encoded)

y_pred_xgb = xgb_model.predict(X_test)
y_pred_xgb = le.inverse_transform(y_pred_xgb)

print("XGBoost Results:")
print(classification_report(y_test, y_pred_xgb))

In [ ]:


import pickle

pickle.dump(rf_model, open("../models/rf_model.pkl", "wb"))
pickle.dump(xgb_model, open("../models/xgb_model.pkl", "wb"))
pickle.dump(le, open("../models/label_encoder.pkl", "wb"))

print("Models saved!")

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2]
}

xgb_tuned = XGBClassifier(random_state=42, eval_metric='mlogloss')

grid_search = GridSearchCV(xgb_tuned, param_grid, cv=3, scoring='f1_weighted', n_jobs=-1)
grid_search.fit(X_train, y_train_encoded)

print("Best parameters:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

In [ ]:
# Train final tuned XGBoost
xgb_final = XGBClassifier(
    learning_rate=0.1,
    max_depth=3,
    n_estimators=200,
    random_state=42,
    eval_metric='mlogloss'
)
xgb_final.fit(X_train, y_train_encoded)

y_pred_final = xgb_final.predict(X_test)
y_pred_final = le.inverse_transform(y_pred_final)

print("Tuned XGBoost Results:")
print(classification_report(y_test, y_pred_final))

In [ ]:
pickle.dump(xgb_final, open("../models/xgb_final.pkl", "wb"))
pickle.dump(le, open("../models/label_encoder.pkl", "wb"))
print("Final model saved!")